<a href="https://colab.research.google.com/github/kayess007/CMSC_6950_Project/blob/main/COLABRAGPROJ.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install sentence-transformers faiss-cpu langchain_community replicate

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from langchain_community.llms import Replicate
import os
from google.colab import userdata


In [ ]:
api_token = userdata.get('REPLICATE_api_token')
os.environ["REPLICATE_API_TOKEN"] = api_token
model = "ibm-granite/granite-3.2-8b-instruct"
output = Replicate(
model=model,
replicate_api_token=api_token,
)


In [ ]:
user_query = input("Ask a question: ")
initial_response = output.invoke(user_query)
print("\nInitial Response (No RAG):\n")

Ask a question: Hi. I’m a citizen of Romania. Do I need a Schengen visa if I travel to France?

Initial Response (No RAG):



In [ ]:
from google.colab import files
uploaded = files.upload()
doc_path = list(uploaded.keys())[0]
with open(doc_path, 'r', encoding='utf-8') as f:
 document_text = f.read()
def split_into_chunks(text, chunk_size=300, overlap=50):
 chunks = []
 start = 0
 while start < len(text):
  end = start + chunk_size
 chunks.append(text[start:end])
 start += chunk_size - overlap
 return chunks
chunks = split_into_chunks(document_text)
print(f"Document split into {len(chunks)} chunks.")

Saving External_KB_for_RAG.txt to External_KB_for_RAG (1).txt


In [ ]:
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embed_model.encode(chunks)
dimension = embeddings.shape[1]
# Initialize FAISS index
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)
# Keep chunks for later retrieval
stored_chunks = chunks
print(f"Stored {len(stored_chunks)} embeddings in FAISS index.")

In [ ]:
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embed_model.encode(chunks)
dimension = embeddings.shape[1]
# Initialize FAISS index
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)
# Keep chunks for later retrieval
stored_chunks = chunks
print(f"Stored {len(stored_chunks)} embeddings in FAISS index.")

In [ ]:
from IPython.display import display, Markdown
display(Markdown(f"""
# Original Query
**{user_query}**
--------------
# Without RAG
{initial_response}
------------------
# With RAG (Context-Augmented)
{rag_response}
"""))